In [ ]:
# # -*- coding: utf-8 -*-
# import os
# import re
# from pathlib import Path
# from typing import Iterable, Dict, Tuple, List
# import pandas as pd
# import matplotlib.pyplot as plt
# # 
# from pptx import Presentation
# from pptx.util import Inches, Pt
# import os, tempfile
# from concurrent.futures import ProcessPoolExecutor, as_completed
# try:
#     import psutil
# except Exception:
#     psutil = None

# os.chdir("..")

# # ------------------------------- Config -------------------------------- #

# WANTED_MEASURE_INDEXES = ["FEV1", "FVC", "FEV1/FVC", "DLCO", "DLCO/VA"]

# MEASURE_INDEX_COLORS: Dict[str, str] = {
#     "FEV1": "tab:blue",
#     "FVC": "tab:orange",
#     "FEV1/FVC": "tab:green",
#     "DLCO": "tab:red",
#     "DLCO/VA": "tab:purple",
# }

# RPT_VALUE_TYPE_MARKERS: Dict[str, str] = {
#     "meas": "o",   # absolute
#     "pred": "D",   # absolute
#     "%pred": "s",  # percentage
#     "%chg": "^",   # percentage
#     "raw": "x",
# }

# DEFAULT_TEST_TYPES = ["PFT", "Bronchodilator", "CO Diffusing"]

# # ----------------------------- Parsers ---------------------------------- #

# _MEASUREINDEX_PAT = re.compile(r"(FEV1/FVC|FEV1|FVC|DLCO/VA|DLCO)", re.IGNORECASE)
# _RPT_PATTERNS = {
#     "%pred": re.compile(r"%\s*pred", re.IGNORECASE),
#     "meas":  re.compile(r"\bmeas(?:ured)?\b", re.IGNORECASE),
#     "pred":  re.compile(r"\bpred(?!\s*%)\b", re.IGNORECASE),
#     "%chg":  re.compile(r"%\s*chg", re.IGNORECASE),
# }
# _TESTTYPE_PATTERNS: List[Tuple[re.Pattern, str]] = [
#     (re.compile(r"\bbronchodilator\b", re.IGNORECASE), "Bronchodilator"),
#     (re.compile(r"\bPFT\b", re.IGNORECASE), "PFT"),
#     (re.compile(r"\bCO\s*Diffusing\b|\bDLCO\b", re.IGNORECASE), "CO Diffusing"),
# ]


# def parse_measure_index(name: str) -> str:
#     m = _MEASUREINDEX_PAT.search(name or "")
#     return m.group(1).upper() if m else "OTHER"


# def parse_rpt_value_type(name: str) -> str:
#     text = name or ""
#     for key, pat in _RPT_PATTERNS.items():
#         if pat.search(text):
#             return key
#     return "raw"


# def parse_test_type(name: str) -> str:
#     text = name or ""
#     for pat, label in _TESTTYPE_PATTERNS:
#         if pat.search(text):
#             return label
#     return "Other"


# # ----------------------------- Pipeline --------------------------------- #

# def load_and_prepare(path_csv: Path) -> pd.DataFrame:
#     """Load merged CSV and add parsed columns; filter to wanted measure indexes."""
#     df = pd.read_csv(path_csv, encoding="utf-8-sig")
#     df["Reception Date"] = pd.to_datetime(df["Reception Date"], format="%Y%m%d", errors="coerce")
#     df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")
#     df["Result item name"] = df["Result item name"].astype("string").fillna("")

#     df["measure_index"] = df["Result item name"].apply(parse_measure_index)
#     df["rpt_value_type"] = df["Result item name"].apply(parse_rpt_value_type)
#     df["test_type"] = df["Result item name"].apply(parse_test_type)

#     sub = (
#         df[df["measure_index"].isin(WANTED_MEASURE_INDEXES)]
#         .dropna(subset=["Reception Date", "Result Numerical Value"])
#         .copy()
#     )
#     return sub


# def plot_patient_context(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     test_type: str,
#     out_dir: Path,
#     measure_index_colors: Dict[str, str] = MEASURE_INDEX_COLORS,
#     rpt_value_type_markers: Dict[str, str] = RPT_VALUE_TYPE_MARKERS,
# ) -> None:
#     """Plot one patient × one test_type with twin y-axes and legend below."""
#     dfx = dfx[(dfx["Patient Number"] == patient_id) & (dfx["test_type"] == test_type)]
#     if dfx.empty:
#         return

#     dfx = dfx.sort_values("Reception Date")

#     fig, ax_abs = plt.subplots(figsize=(12, 6))
#     ax_pct = ax_abs.twinx()  # right axis for percentages

#     handles_abs, handles_pct = [], []
#     variants_present = sorted(dfx["rpt_value_type"].dropna().unique())
#     for mi in WANTED_MEASURE_INDEXES:
#         for rvt in variants_present:
#             dft = dfx[(dfx["measure_index"] == mi) & (dfx["rpt_value_type"] == rvt)]
#             if dft.empty:
#                 continue

#             color = measure_index_colors.get(mi, "black")
#             marker = rpt_value_type_markers.get(rvt, ".")

#             if rvt in ("meas", "pred"):  # absolute values
#                 h, = ax_abs.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_abs.append(h)
#             elif rvt in ("%pred", "%chg"):  # percentages
#                 h, = ax_pct.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linestyle="--", linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_pct.append(h)
#             else:  # unknown → put on absolute axis
#                 h, = ax_abs.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_abs.append(h)

#     ax_abs.set_title(f"Patient {patient_id} — {test_type}: All Variants by measure_index")
#     ax_abs.set_xlabel("Reception Date")
#     ax_abs.set_ylabel("Absolute value (meas, pred)")
#     ax_pct.set_ylabel("Percent (%pred, %chg)")
#     ax_abs.grid(True, linestyle="--", alpha=0.35)
#     for t in ax_abs.get_xticklabels():
#         t.set_rotation(20)

#     y0, y1 = ax_pct.get_ylim()
#     if y1 < 100:
#         ax_pct.set_ylim(0, 110)
#     else:
#         ax_pct.set_ylim(0, min(140, y1))

#     handles = handles_abs + handles_pct
#     if handles:
#         fig.legend(
#             handles,
#             [h.get_label() for h in handles],
#             loc="lower center",
#             bbox_to_anchor=(0.5, -0.12),
#             ncol=3,
#             fontsize=9,
#             frameon=False,
#         )
#         fig.subplots_adjust(bottom=0.22)

#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_{test_type.replace(' ', '_')}_all_variants_twin.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)


# def plot_patients(
#     df: pd.DataFrame,
#     patient_ids: Iterable[int],
#     test_types: Iterable[str] = DEFAULT_TEST_TYPES,
#     out_dir: Path = Path("data/interim/pft_plots_all_variants"),
#     make_ppt: bool = False
# ) -> None:
#     """Generate all plots (patients × test_types) and a PPT per patient."""
#     out_dir = Path(out_dir)
#     for pid in patient_ids:
#         # 1) Make plots
#         for tt in test_types:
#             plot_patient_context(df, pid, tt, out_dir)
#         # 2) Make PPT for this patient
#             make_patient_ppt(
#                 patient_id=pid,
#                 test_types=test_types,
#                 plots_dir=out_dir,
#                 ppt_out_path=out_dir / "ppt" / f"patient_{pid}_pft_summary.pptx",
#             )


# def make_patient_ppt(
#     patient_id: int,
#     test_types: Iterable[str],
#     plots_dir: Path,
#     ppt_out_path: Path | None = None,
#     title: str | None = None,
# ) -> Path:
#     """
#     Create a PPTX with all plots for a single patient.
#     Expects images saved by `plot_patient_context` like:
#       patient_{id}_{test_type}_all_variants_twin.png
#     """
#     plots_dir = Path(plots_dir)
#     if ppt_out_path is None:
#         ppt_out_dir = plots_dir / "ppt"
#         ppt_out_dir.mkdir(parents=True, exist_ok=True)
#         ppt_out_path = ppt_out_dir / f"patient_{patient_id}_pft_summary.pptx"
#     else:
#         Path(ppt_out_path).parent.mkdir(parents=True, exist_ok=True)

#     prs = Presentation()
#     # Ensure 16:9 (inches)
#     prs.slide_width  = Inches(13.33)
#     prs.slide_height = Inches(7.5)

#     # Title slide
#     title_layout = prs.slide_layouts[0]  # Title
#     slide = prs.slides.add_slide(title_layout)
#     slide.shapes.title.text = title or f"Patient {patient_id} — Pulmonary Function Summary"
#     slide.placeholders[1].text = "Generated from merged PFT data (meas/pred/%pred/%chg)"

#     # Add one slide per test type (if image exists)
#     blank_layout = prs.slide_layouts[6]  # Blank
#     found_any = False
#     for tt in test_types:
#         img_name = f"patient_{patient_id}_{tt.replace(' ', '_')}_all_variants_twin.png"
#         img_path = plots_dir / img_name
#         if not img_path.exists():
#             continue

#         found_any = True
#         slide = prs.slides.add_slide(blank_layout)

#         # Optional slide title (simple text box)
#         left = Inches(0.5)
#         top = Inches(0.3)
#         width = Inches(12.33)
#         height = Inches(0.6)
#         title_box = slide.shapes.add_textbox(left, top, width, height)
#         p = title_box.text_frame.paragraphs[0]
#         p.text = f"{tt}: All Variants by measure_index"
#         p.font.size = Pt(24)

#         # Place the image centered below title
#         img_top = Inches(1.1)
#         img_left = Inches(0.5)
#         img_width = Inches(12.33)
#         img_height = Inches(6)
#         slide.shapes.add_picture(str(img_path), img_left, img_top, height=img_height) # width=img_width

#     if not found_any:
#         # Still save a minimal deck to signal “no plots”
#         note_slide = prs.slides.add_slide(blank_layout)
#         tb = note_slide.shapes.add_textbox(Inches(1), Inches(2.5), Inches(10), Inches(1))
#         tb.text_frame.text = f"No plots found for patient {patient_id} in {plots_dir}"

#     prs.save(ppt_out_path)
#     return ppt_out_path


# def main():
#     data_path = Path("data/processed/pft_merged.csv")
#     out_dir = Path("data/interim/pft_plots_all_variants")

    
#     df = load_and_prepare(data_path)
#     patient_ids = [886482, 1207865, 1452945, 611957, 965594]
#     # patient_ids = df["Patient Number"].unique()
#     print("Total patients:", len(patient_ids))
#     plot_patients(df, patient_ids, DEFAULT_TEST_TYPES, out_dir, make_ppt=True)

# main()
# # if __name__ == "__main__":
# #     main()


C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_72204\554934186.py:83: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_csv, encoding="utf-8-sig")


Total patients: 5


In [ ]:
# # -*- coding: utf-8 -*-
# import os
# import re
# from pathlib import Path
# from typing import Iterable, Dict, Tuple, List
# import pandas as pd
# import matplotlib.pyplot as plt

# from pptx import Presentation
# from pptx.util import Inches, Pt
# import os, tempfile
# from concurrent.futures import ProcessPoolExecutor, as_completed
# try:
#     import psutil
# except Exception:
#     psutil = None

# os.chdir("..")

# def _init_worker(df, test_types, out_dir, make_ppt):
#     # Keep NumPy/MKL/OpenBLAS single-threaded inside each worker
#     os.environ.setdefault("MKL_NUM_THREADS", "1")
#     os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
#     os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
#     os.environ.setdefault("OMP_NUM_THREADS", "1")

#     # Stash globals in the child so each task doesn't re-pickle everything
#     global GDF, G_TEST_TYPES, G_OUT_DIR, G_MAKE_PPT
#     GDF = df
#     G_TEST_TYPES = list(test_types)
#     G_OUT_DIR = Path(out_dir)
#     G_MAKE_PPT = bool(make_ppt)

# def _patient_task(pid: int):
#     import matplotlib
#     os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), f"mpl-{os.getpid()}"))
#     matplotlib.use("Agg")

#     try:
#         for tt in G_TEST_TYPES:
#             plot_patient_context(GDF, pid, tt, G_OUT_DIR)
#         if G_MAKE_PPT:
#             (G_OUT_DIR / "ppt").mkdir(parents=True, exist_ok=True)
#             make_patient_ppt(
#                 patient_id=pid,
#                 test_types=G_TEST_TYPES,
#                 plots_dir=G_OUT_DIR,
#                 ppt_out_path=G_OUT_DIR / "ppt" / f"patient_{pid}_pft_summary.pptx",
#             )
#         return pid, "ok"
#     except Exception as e:
#         return pid, f"error: {e}"


# # ------------------------------- Config -------------------------------- #

# WANTED_MEASURE_INDEXES = ["FEV1", "FVC", "FEV1/FVC", "DLCO", "DLCO/VA"]

# MEASURE_INDEX_COLORS: Dict[str, str] = {
#     "FEV1": "tab:blue",
#     "FVC": "tab:orange",
#     "FEV1/FVC": "tab:green",
#     "DLCO": "tab:red",
#     "DLCO/VA": "tab:purple",
# }

# RPT_VALUE_TYPE_MARKERS: Dict[str, str] = {
#     "meas": "o",   # absolute
#     "pred": "D",   # absolute
#     "%pred": "s",  # percentage
#     "%chg": "^",   # percentage
#     "raw": "x",
# }

# DEFAULT_TEST_TYPES = ["PFT", "Bronchodilator", "CO Diffusing"]

# # ----------------------------- Parsers ---------------------------------- #

# _MEASUREINDEX_PAT = re.compile(r"(FEV1/FVC|FEV1|FVC|DLCO/VA|DLCO)", re.IGNORECASE)
# _RPT_PATTERNS = {
#     "%pred": re.compile(r"%\s*pred", re.IGNORECASE),
#     "meas":  re.compile(r"\bmeas(?:ured)?\b", re.IGNORECASE),
#     "pred":  re.compile(r"\bpred(?!\s*%)\b", re.IGNORECASE),
#     "%chg":  re.compile(r"%\s*chg", re.IGNORECASE),
# }
# _TESTTYPE_PATTERNS: List[Tuple[re.Pattern, str]] = [
#     (re.compile(r"\bbronchodilator\b", re.IGNORECASE), "Bronchodilator"),
#     (re.compile(r"\bPFT\b", re.IGNORECASE), "PFT"),
#     (re.compile(r"\bCO\s*Diffusing\b|\bDLCO\b", re.IGNORECASE), "CO Diffusing"),
# ]


# def parse_measure_index(name: str) -> str:
#     m = _MEASUREINDEX_PAT.search(name or "")
#     return m.group(1).upper() if m else "OTHER"


# def parse_rpt_value_type(name: str) -> str:
#     text = name or ""
#     for key, pat in _RPT_PATTERNS.items():
#         if pat.search(text):
#             return key
#     return "raw"


# def parse_test_type(name: str) -> str:
#     text = name or ""
#     for pat, label in _TESTTYPE_PATTERNS:
#         if pat.search(text):
#             return label
#     return "Other"


# # ----------------------------- Pipeline --------------------------------- #

# def load_and_prepare(path_csv: Path) -> pd.DataFrame:
#     """Load merged CSV and add parsed columns; filter to wanted measure indexes."""
#     df = pd.read_csv(path_csv, encoding="utf-8-sig")
#     df["Reception Date"] = pd.to_datetime(df["Reception Date"], format="%Y%m%d", errors="coerce")
#     df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")
#     df["Result item name"] = df["Result item name"].astype("string").fillna("")

#     df["measure_index"] = df["Result item name"].apply(parse_measure_index)
#     df["rpt_value_type"] = df["Result item name"].apply(parse_rpt_value_type)
#     df["test_type"] = df["Result item name"].apply(parse_test_type)

#     sub = (
#         df[df["measure_index"].isin(WANTED_MEASURE_INDEXES)]
#         .dropna(subset=["Reception Date", "Result Numerical Value"])
#         .copy()
#     )
#     return sub


# def plot_patient_context(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     test_type: str,
#     out_dir: Path,
#     measure_index_colors: Dict[str, str] = MEASURE_INDEX_COLORS,
#     rpt_value_type_markers: Dict[str, str] = RPT_VALUE_TYPE_MARKERS,
# ) -> None:
#     """Plot one patient × one test_type with twin y-axes and legend below."""
#     dfx = dfx[(dfx["Patient Number"] == patient_id) & (dfx["test_type"] == test_type)]
#     if dfx.empty:
#         return

#     dfx = dfx.sort_values("Reception Date")

#     fig, ax_abs = plt.subplots(figsize=(12, 6))
#     ax_pct = ax_abs.twinx()  # right axis for percentages

#     handles_abs, handles_pct = [], []
#     variants_present = sorted(dfx["rpt_value_type"].dropna().unique())
#     for mi in WANTED_MEASURE_INDEXES:
#         for rvt in variants_present:
#             dft = dfx[(dfx["measure_index"] == mi) & (dfx["rpt_value_type"] == rvt)]
#             if dft.empty:
#                 continue

#             color = measure_index_colors.get(mi, "black")
#             marker = rpt_value_type_markers.get(rvt, ".")

#             if rvt in ("meas", "pred"):  # absolute values
#                 h, = ax_abs.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_abs.append(h)
#             elif rvt in ("%pred", "%chg"):  # percentages
#                 h, = ax_pct.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linestyle="--", linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_pct.append(h)
#             else:  # unknown → put on absolute axis
#                 h, = ax_abs.plot(
#                     dft["Reception Date"],
#                     dft["Result Numerical Value"],
#                     marker=marker, color=color, linewidth=1.6,
#                     label=f"{mi} [{rvt}] (n={len(dft)})"
#                 )
#                 handles_abs.append(h)

#     ax_abs.set_title(f"Patient {patient_id} — {test_type}: All Variants by measure_index")
#     ax_abs.set_xlabel("Reception Date")
#     ax_abs.set_ylabel("Absolute value (meas, pred)")
#     ax_pct.set_ylabel("Percent (%pred, %chg)")
#     ax_abs.grid(True, linestyle="--", alpha=0.35)
#     for t in ax_abs.get_xticklabels():
#         t.set_rotation(20)

#     y0, y1 = ax_pct.get_ylim()
#     if y1 < 100:
#         ax_pct.set_ylim(0, 110)
#     else:
#         ax_pct.set_ylim(0, min(140, y1))

#     handles = handles_abs + handles_pct
#     if handles:
#         fig.legend(
#             handles,
#             [h.get_label() for h in handles],
#             loc="lower center",
#             bbox_to_anchor=(0.5, -0.12),
#             ncol=3,
#             fontsize=9,
#             frameon=False,
#         )
#         fig.subplots_adjust(bottom=0.22)

#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_{test_type.replace(' ', '_')}_all_variants_twin.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)


# def plot_patients(
#     df: pd.DataFrame,
#     patient_ids: Iterable[int],
#     test_types: Iterable[str] = DEFAULT_TEST_TYPES,
#     out_dir: Path = Path("data/interim/pft_plots_all_variants"),
#     make_ppt: bool = False,
#     n_jobs: int | None = None,
# ) -> None:
#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)

#     # Choose worker count (prefer physical cores)
#     if n_jobs is None:
#         if psutil is not None:
#             n_phys = psutil.cpu_count(logical=False) or (os.cpu_count() or 1) // 2
#         else:
#             n_phys = (os.cpu_count() or 1) // 2
#         n_jobs = max(1, n_phys)

#     with ProcessPoolExecutor(
#         max_workers=n_jobs,
#         initializer=_init_worker,
#         initargs=(df, list(test_types), out_dir, make_ppt),
#     ) as ex:
#         futures = [ex.submit(_patient_task, int(pid)) for pid in patient_ids]
#         for fut in as_completed(futures):
#             pid, status = fut.result()
#             print(f"[patient {pid}] {status}")


# def make_patient_ppt(
#     patient_id: int,
#     test_types: Iterable[str],
#     plots_dir: Path,
#     ppt_out_path: Path | None = None,
#     title: str | None = None,
# ) -> Path:
#     """
#     Create a PPTX with all plots for a single patient.
#     Expects images saved by `plot_patient_context` like:
#       patient_{id}_{test_type}_all_variants_twin.png
#     """
#     plots_dir = Path(plots_dir)
#     if ppt_out_path is None:
#         ppt_out_dir = plots_dir / "ppt"
#         ppt_out_dir.mkdir(parents=True, exist_ok=True)
#         ppt_out_path = ppt_out_dir / f"patient_{patient_id}_pft_summary.pptx"
#     else:
#         Path(ppt_out_path).parent.mkdir(parents=True, exist_ok=True)

#     prs = Presentation()
#     # Ensure 16:9 (inches)
#     prs.slide_width  = Inches(13.33)
#     prs.slide_height = Inches(7.5)

#     # Title slide
#     title_layout = prs.slide_layouts[0]  # Title
#     slide = prs.slides.add_slide(title_layout)
#     slide.shapes.title.text = title or f"Patient {patient_id} — Pulmonary Function Summary"
#     slide.placeholders[1].text = "Generated from merged PFT data (meas/pred/%pred/%chg)"

#     # Add one slide per test type (if image exists)
#     blank_layout = prs.slide_layouts[6]  # Blank
#     found_any = False
#     for tt in test_types:
#         img_name = f"patient_{patient_id}_{tt.replace(' ', '_')}_all_variants_twin.png"
#         img_path = plots_dir / img_name
#         if not img_path.exists():
#             continue

#         found_any = True
#         slide = prs.slides.add_slide(blank_layout)

#         # Optional slide title (simple text box)
#         left = Inches(0.5)
#         top = Inches(0.3)
#         width = Inches(12.33)
#         height = Inches(0.6)
#         title_box = slide.shapes.add_textbox(left, top, width, height)
#         p = title_box.text_frame.paragraphs[0]
#         p.text = f"{tt}: All Variants by measure_index"
#         p.font.size = Pt(24)

#         # Place the image centered below title
#         img_top = Inches(1.1)
#         img_left = Inches(0.5)
#         img_width = Inches(12.33)
#         img_height = Inches(6)
#         slide.shapes.add_picture(str(img_path), img_left, img_top, height=img_height) # width=img_width

#     if not found_any:
#         # Still save a minimal deck to signal “no plots”
#         note_slide = prs.slides.add_slide(blank_layout)
#         tb = note_slide.shapes.add_textbox(Inches(1), Inches(2.5), Inches(10), Inches(1))
#         tb.text_frame.text = f"No plots found for patient {patient_id} in {plots_dir}"

#     prs.save(ppt_out_path)
#     return ppt_out_path


# def main():
#     data_path = Path("data/processed/pft_merged.csv")
#     out_dir = Path("data/interim/pft_plots_all_variants")

    
#     df = load_and_prepare(data_path)
#     patient_ids = [886482, 1207865, 1452945, 611957, 965594]
#     # patient_ids = patient_ids = df["Patient Number"].dropna().astype(int).unique()
#     print("Total patients:", len(patient_ids))
#     # plot_patients(df, patient_ids, DEFAULT_TEST_TYPES, out_dir, make_ppt=True)

#     # Auto-detect physical cores (or set n_jobs=32)
#     if psutil is not None:
#         n_jobs = psutil.cpu_count(logical=False) or (os.cpu_count() or 1) // 2
#     else:
#         n_jobs = (os.cpu_count() or 1) // 2
#     n_jobs = max(1, n_jobs)

#     plot_patients(df, patient_ids, DEFAULT_TEST_TYPES, out_dir, make_ppt=True, n_jobs=n_jobs)

# main()
# # if __name__ == "__main__":
# #     main()


C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_59228\1601193402.py:64: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_csv, encoding="utf-8-sig")


Total patients: 5


In [1]:
# -*- coding: utf-8 -*-
import os
import re
from pathlib import Path
from typing import Iterable, Dict, Tuple, List
import pandas as pd
import matplotlib.pyplot as plt

from pptx import Presentation
from pptx.util import Inches, Pt
import os, tempfile
from concurrent.futures import ProcessPoolExecutor, as_completed
try:
    import psutil
except Exception:
    psutil = None

os.chdir("..")

In [ ]:


def _init_worker(test_types, out_dir, make_ppt):
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    os.environ.setdefault("OMP_NUM_THREADS", "1")

    # Stash globals in the child so each task doesn't re-pickle everything
    global G_TEST_TYPES, G_OUT_DIR, G_MAKE_PPT
    G_TEST_TYPES = list(test_types)
    G_OUT_DIR = Path(out_dir)
    G_MAKE_PPT = bool(make_ppt)


def _patient_task(payload):
    pid, dfx = payload  # <-- unpack the (pid, slice) pair
    import matplotlib
    os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), f"mpl-{os.getpid()}"))
    matplotlib.use("Agg")

    try:
        for tt in G_TEST_TYPES:
            plot_patient_context(dfx, pid, tt, G_OUT_DIR)   # <-- use dfx (not GDF)
        # (Optional) do NOT build PPT here; do it after the pool to speed things up
        if G_MAKE_PPT:
            (G_OUT_DIR / "ppt").mkdir(parents=True, exist_ok=True)
            make_patient_ppt(
                patient_id=pid,
                test_types=G_TEST_TYPES,
                plots_dir=G_OUT_DIR,
                ppt_out_path=G_OUT_DIR / "ppt" / f"patient_{pid}_pft_summary.pptx",
            )
        return pid, "ok"
    except Exception as e:
        return pid, f"error: {e}"


# ------------------------------- Config -------------------------------- #

WANTED_MEASURE_INDEXES = ["FEV1", "FVC", "FEV1/FVC", "DLCO", "DLCO/VA"]

MEASURE_INDEX_COLORS: Dict[str, str] = {
    "FEV1": "tab:blue",
    "FVC": "tab:orange",
    "FEV1/FVC": "tab:green",
    "DLCO": "tab:red",
    "DLCO/VA": "tab:purple",
}

RPT_VALUE_TYPE_MARKERS: Dict[str, str] = {
    "meas": "o",   # absolute
    "pred": "D",   # absolute
    "%pred": "s",  # percentage
    "%chg": "^",   # percentage
    "raw": "x",
}

DEFAULT_TEST_TYPES = ["PFT", "Bronchodilator", "CO Diffusing"]

# ----------------------------- Parsers ---------------------------------- #

_MEASUREINDEX_PAT = re.compile(r"(FEV1/FVC|FEV1|FVC|DLCO/VA|DLCO)", re.IGNORECASE)
_RPT_PATTERNS = {
    "%pred": re.compile(r"%\s*pred", re.IGNORECASE),
    "meas":  re.compile(r"\bmeas(?:ured)?\b", re.IGNORECASE),
    "pred":  re.compile(r"\bpred(?!\s*%)\b", re.IGNORECASE),
    "%chg":  re.compile(r"%\s*chg", re.IGNORECASE),
}
_TESTTYPE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\bbronchodilator\b", re.IGNORECASE), "Bronchodilator"),
    (re.compile(r"\bPFT\b", re.IGNORECASE), "PFT"),
    (re.compile(r"\bCO\s*Diffusing\b|\bDLCO\b", re.IGNORECASE), "CO Diffusing"),
]


def parse_measure_index(name: str) -> str:
    m = _MEASUREINDEX_PAT.search(name or "")
    return m.group(1).upper() if m else "OTHER"


def parse_rpt_value_type(name: str) -> str:
    text = name or ""
    for key, pat in _RPT_PATTERNS.items():
        if pat.search(text):
            return key
    return "raw"


def parse_test_type(name: str) -> str:
    text = name or ""
    for pat, label in _TESTTYPE_PATTERNS:
        if pat.search(text):
            return label
    return "Other"


# ----------------------------- Pipeline --------------------------------- #

def load_and_prepare(path_csv: Path) -> pd.DataFrame:
    """Load merged CSV and add parsed columns; filter to wanted measure indexes."""
    df = pd.read_csv(path_csv, encoding="utf-8-sig")
    df["Reception Date"] = pd.to_datetime(df["Reception Date"], format="%Y%m%d", errors="coerce")
    df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")
    df["Result item name"] = df["Result item name"].astype("string").fillna("")

    df["measure_index"] = df["Result item name"].apply(parse_measure_index)
    df["rpt_value_type"] = df["Result item name"].apply(parse_rpt_value_type)
    df["test_type"] = df["Result item name"].apply(parse_test_type)

    sub = (
        df[df["measure_index"].isin(WANTED_MEASURE_INDEXES)]
        .dropna(subset=["Reception Date", "Result Numerical Value"])
        .copy()
    )
    return sub


def plot_patient_context(
    dfx: pd.DataFrame,
    patient_id: int,
    test_type: str,
    out_dir: Path,
    measure_index_colors: Dict[str, str] = MEASURE_INDEX_COLORS,
    rpt_value_type_markers: Dict[str, str] = RPT_VALUE_TYPE_MARKERS,
) -> None:
    """Plot one patient × one test_type with twin y-axes and legend below."""
    dfx = dfx[(dfx["Patient Number"] == patient_id) & (dfx["test_type"] == test_type)]
    if dfx.empty:
        return

    dfx = dfx.sort_values("Reception Date")

    fig, ax_abs = plt.subplots(figsize=(12, 6))
    ax_pct = ax_abs.twinx()  # right axis for percentages

    handles_abs, handles_pct = [], []
    variants_present = sorted(dfx["rpt_value_type"].dropna().unique())
    for mi in WANTED_MEASURE_INDEXES:
        for rvt in variants_present:
            dft = dfx[(dfx["measure_index"] == mi) & (dfx["rpt_value_type"] == rvt)]
            if dft.empty:
                continue

            color = measure_index_colors.get(mi, "black")
            marker = rpt_value_type_markers.get(rvt, ".")

            if rvt in ("meas", "pred"):  # absolute values
                h, = ax_abs.plot(
                    dft["Reception Date"],
                    dft["Result Numerical Value"],
                    marker=marker, color=color, linewidth=1.6,
                    label=f"{mi} [{rvt}] (n={len(dft)})"
                )
                handles_abs.append(h)
            elif rvt in ("%pred", "%chg"):  # percentages
                h, = ax_pct.plot(
                    dft["Reception Date"],
                    dft["Result Numerical Value"],
                    marker=marker, color=color, linestyle="--", linewidth=1.6,
                    label=f"{mi} [{rvt}] (n={len(dft)})"
                )
                handles_pct.append(h)
            else:  # unknown → put on absolute axis
                h, = ax_abs.plot(
                    dft["Reception Date"],
                    dft["Result Numerical Value"],
                    marker=marker, color=color, linewidth=1.6,
                    label=f"{mi} [{rvt}] (n={len(dft)})"
                )
                handles_abs.append(h)

    ax_abs.set_title(f"Patient {patient_id} — {test_type}: All Variants by measure_index")
    ax_abs.set_xlabel("Reception Date")
    ax_abs.set_ylabel("Absolute value (meas, pred)")
    ax_pct.set_ylabel("Percent (%pred, %chg)")
    ax_abs.grid(True, linestyle="--", alpha=0.35)
    for t in ax_abs.get_xticklabels():
        t.set_rotation(20)

    y0, y1 = ax_pct.get_ylim()
    if y1 < 100:
        ax_pct.set_ylim(0, 110)
    else:
        ax_pct.set_ylim(0, min(140, y1))

    handles = handles_abs + handles_pct
    if handles:
        fig.legend(
            handles,
            [h.get_label() for h in handles],
            loc="lower center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=3,
            fontsize=9,
            frameon=False,
        )
        fig.subplots_adjust(bottom=0.22)

    out_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"patient_{patient_id}_{test_type.replace(' ', '_')}_all_variants_twin.png"
    fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
    plt.close(fig)


# def plot_patients(
#     df: pd.DataFrame,
#     patient_ids: Iterable[int],
#     test_types: Iterable[str] = DEFAULT_TEST_TYPES,
#     out_dir: Path = Path("data/interim/pft_plots_all_variants"),
#     make_ppt: bool = False,
#     n_jobs: int | None = None,
# ) -> None:
#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)

#     # Choose worker count (prefer physical cores)
#     if n_jobs is None:
#         if psutil is not None:
#             n_phys = psutil.cpu_count(logical=False) or (os.cpu_count() or 1) // 2
#         else:
#             n_phys = (os.cpu_count() or 1) // 2
#         n_jobs = max(1, n_phys)

#     with ProcessPoolExecutor(
#     max_workers=n_jobs,
#     initializer=_init_worker,
#     initargs=(list(test_types), out_dir, make_ppt),
#     ) as ex:
#         futures = [ex.submit(_patient_task, (pid, by_pid[pid])) for pid in patient_ids]
#         for fut in as_completed(futures):
#             pid, status = fut.result()
#             print(f"[patient {pid}] {status}")

    # # After workers finish, build PPTs sequentially (fast I/O, no CPU thrash):
    # if make_ppt:
    #     (out_dir / "ppt").mkdir(parents=True, exist_ok=True)
    #     for pid in patient_ids:
    #         make_patient_ppt(
    #             patient_id=pid,
    #             test_types=test_types,
    #             plots_dir=out_dir,
    #             ppt_out_path=out_dir / "ppt" / f"patient_{pid}_pft_summary.pptx",
    #         )

def plot_patients(
    df: pd.DataFrame,
    patient_ids: Iterable[int],
    test_types: Iterable[str] = DEFAULT_TEST_TYPES,
    out_dir: Path = Path("data/interim/pft_plots_all_variants"),
    make_ppt: bool = False,
    n_jobs: int | None = None,
) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 🔴 ADD THIS HERE
    by_pid = {int(pid): grp.copy() for pid, grp in df.groupby("Patient Number")}

    # Choose worker count (prefer physical cores)
    if n_jobs is None:
        if psutil is not None:
            n_phys = psutil.cpu_count(logical=False) or (os.cpu_count() or 1) // 2
        else:
            n_phys = (os.cpu_count() or 1) // 2
        n_jobs = max(1, n_phys)

    with ProcessPoolExecutor(
        max_workers=n_jobs,
        initializer=_init_worker,
        initargs=(list(test_types), out_dir, make_ppt),
    ) as ex:
        futures = [ex.submit(_patient_task, (pid, by_pid[pid])) for pid in patient_ids]
        for fut in as_completed(futures):
            pid, status = fut.result()
            print(f"[patient {pid}] {status}")


def make_patient_ppt(
    patient_id: int,
    test_types: Iterable[str],
    plots_dir: Path,
    ppt_out_path: Path | None = None,
    title: str | None = None,
) -> Path:
    """
    Create a PPTX with all plots for a single patient.
    Expects images saved by `plot_patient_context` like:
      patient_{id}_{test_type}_all_variants_twin.png
    """
    plots_dir = Path(plots_dir)
    if ppt_out_path is None:
        ppt_out_dir = plots_dir / "ppt"
        ppt_out_dir.mkdir(parents=True, exist_ok=True)
        ppt_out_path = ppt_out_dir / f"patient_{patient_id}_pft_summary.pptx"
    else:
        Path(ppt_out_path).parent.mkdir(parents=True, exist_ok=True)

    prs = Presentation()
    # Ensure 16:9 (inches)
    prs.slide_width  = Inches(13.33)
    prs.slide_height = Inches(7.5)

    # Title slide
    title_layout = prs.slide_layouts[0]  # Title
    slide = prs.slides.add_slide(title_layout)
    slide.shapes.title.text = title or f"Patient {patient_id} — Pulmonary Function Summary"
    slide.placeholders[1].text = "Generated from merged PFT data (meas/pred/%pred/%chg)"

    # Add one slide per test type (if image exists)
    blank_layout = prs.slide_layouts[6]  # Blank
    found_any = False
    for tt in test_types:
        img_name = f"patient_{patient_id}_{tt.replace(' ', '_')}_all_variants_twin.png"
        img_path = plots_dir / img_name
        if not img_path.exists():
            continue

        found_any = True
        slide = prs.slides.add_slide(blank_layout)

        # Optional slide title (simple text box)
        left = Inches(0.5)
        top = Inches(0.3)
        width = Inches(12.33)
        height = Inches(0.6)
        title_box = slide.shapes.add_textbox(left, top, width, height)
        p = title_box.text_frame.paragraphs[0]
        p.text = f"{tt}: All Variants by measure_index"
        p.font.size = Pt(24)

        # Place the image centered below title
        img_top = Inches(1.1)
        img_left = Inches(0.5)
        img_width = Inches(12.33)
        img_height = Inches(6)
        slide.shapes.add_picture(str(img_path), img_left, img_top, height=img_height) # width=img_width

    if not found_any:
        # Still save a minimal deck to signal “no plots”
        note_slide = prs.slides.add_slide(blank_layout)
        tb = note_slide.shapes.add_textbox(Inches(1), Inches(2.5), Inches(10), Inches(1))
        tb.text_frame.text = f"No plots found for patient {patient_id} in {plots_dir}"

    prs.save(ppt_out_path)
    return ppt_out_path


def main():
    data_path = Path("data/processed/pft_merged.csv")
    out_dir = Path("data/interim/pft_plots_all_variants")

    
    df = load_and_prepare(data_path)
    # patient_ids = [886482, 1207865, 1452945, 611957, 965594]
    # patient_ids = patient_ids = df["Patient Number"].dropna().astype(int).unique()

    # Group once; build a dict of tiny per-patient DataFrames
    by_pid = {int(pid): grp.copy() for pid, grp in df.groupby("Patient Number")}
    patient_ids = list(by_pid.keys())
    print("Total patients:", len(patient_ids))
    # plot_patients(df, patient_ids, DEFAULT_TEST_TYPES, out_dir, make_ppt=True)

    # Auto-detect physical cores (or set n_jobs=32)
    if psutil is not None:
        n_jobs = psutil.cpu_count(logical=False) or (os.cpu_count() or 1) // 2
    else:
        n_jobs = (os.cpu_count() or 1) // 2
    n_jobs = max(1, n_jobs)

    plot_patients(df, patient_ids, DEFAULT_TEST_TYPES, out_dir, make_ppt=True, n_jobs=n_jobs)

main()
# if __name__ == "__main__":
#     main()


C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_76548\812020107.py:100: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_csv, encoding="utf-8-sig")


Total patients: 27331
